[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/doxav/astromodel_proving/blob/main/analysis/09_reviewer_response_synthesis.ipynb)

# Step 09 - Reviewer-response synthesis

This notebook integrates Steps 00-08 into reviewer-facing traceability, claim-maturity, manuscript asset tables, and selected reviewer-action gate audits. The selected-action audits create no new biological evidence and run no Step 04 optimizer; they derive objective gate tables from current outputs so unsupported claims can be restricted, upgraded, or kept blocked without silent overclaiming.

In [1]:
from pathlib import Path
import os
import sys
import pandas as pd

PROJECT_ROOT = Path(os.environ.get("ASTROMODEL_PROJECT_ROOT", ".")).resolve()
if not (PROJECT_ROOT / "src").exists():
    current = Path.cwd().resolve()
    PROJECT_ROOT = next((p for p in [current, *current.parents] if (p / "src").exists()), PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT


PosixPath('/home/xav/code/astromodel_proving')

## Selected-action gate audits

In [2]:
from src.reviewer_gate_audits import ReviewerGateAuditConfig, run_reviewer_gate_audits

gate_config = ReviewerGateAuditConfig(write_outputs=True)
gate_result = run_reviewer_gate_audits(PROJECT_ROOT, gate_config)
pd.DataFrame([
    {"artifact": key, "n_rows": len(value), "n_columns": len(value.columns)}
    for key, value in gate_result.items()
]).sort_values("artifact").reset_index(drop=True)


,artifact,n_rows,n_columns
0,K_o_homeostasis_endpoint_audit,17,13
1,all_current_assumption_sensitivity,108,7
2,assumption_gate_audit,3,8
3,cell_specific_identifiability_audit,315,11
4,claim_to_artifact_ledger,6,6
5,constrained_failure_modes,30,13
6,degeneracy_level_table,30,11
7,full_accepted_parameter_audit,18990,27
8,integrated_degeneracy_gate_matrix,30,25
9,intercluster_interpolation_acceptance,390,18


## Selected-action scientific value

In [3]:
selected_summary = pd.read_csv(PROJECT_ROOT / "outputs" / "reviewer_synthesis" / "selected_action_results_summary.csv")
scientific_value = pd.read_csv(PROJECT_ROOT / "outputs" / "reviewer_synthesis" / "selected_action_scientific_value_assessment.csv")
display(selected_summary)
display(scientific_value)


,artifact_key,n_rows,n_columns,non_empty,can_upgrade_claim_directly
0,selected_action_strategy_comparison,3,8,True,False
1,phenotype_robustness_summary,17,13,True,False
2,stratum_support_gate,6,14,True,False
3,prediction_limited_failure_modes,5,15,True,False
4,assumption_gate_audit,3,8,True,False
5,proxy_exclusion_claim_sensitivity,8,11,True,False
6,parameter_semantics_audit,9,11,True,False
7,full_accepted_parameter_audit,18990,27,True,False
8,parameter_interpretation_class_audit,270,34,True,False
9,constrained_failure_modes,30,13,True,False


,artifact_key,n_rows,n_columns,scientific_value_status,rationale
0,selected_action_strategy_comparison,3,8,retain,artifact has rows and explicit gates
1,phenotype_robustness_summary,17,13,retain,artifact has rows and explicit gates
2,stratum_support_gate,6,14,retain,artifact has rows and explicit gates
3,prediction_limited_failure_modes,5,15,retain,artifact has rows and explicit gates
4,assumption_gate_audit,3,8,retain_as_blocker_evidence,assumption gate failures are scientifically im...
5,proxy_exclusion_claim_sensitivity,8,11,retain,artifact has rows and explicit gates
6,parameter_semantics_audit,9,11,retain,artifact has rows and explicit gates
7,full_accepted_parameter_audit,18990,27,retain,artifact has rows and explicit gates
8,parameter_interpretation_class_audit,270,34,retain,artifact has rows and explicit gates
9,constrained_failure_modes,30,13,retain,artifact has rows and explicit gates


In [4]:
from src.step09_reviewer_synthesis import Step09Config, run_step09_reviewer_synthesis

config = Step09Config(write_outputs=True)
result = run_step09_reviewer_synthesis(PROJECT_ROOT, config)
result["analysis_summary"]

{'step_name': 'Step 09 - reviewer-response synthesis',
 'config': {'write_outputs': True},
 'n_reviewer_rows': 7,
 'n_claim_rows': 6,
 'n_manifest_rows': 35,
 'missing_manifest_artifacts': [],
 'final_biological_degeneracy_claim_allowed': False,
 'headline_claim_scope': 'Final degeneracy wording is allowed only if all upstream evidence layers are supported.',
 'elapsed_seconds': 0.020797915996809024}

## R1-R7 traceability

In [5]:
traceability = result["reviewer_traceability_table"]
traceability

,reviewer_id,critique,primary_outputs,evidence_status,claim_boundary,final_biological_degeneracy_claim_allowed
0,R1,degeneracy versus non-identifiability/sloppiness,outputs/identifiability; outputs/mechanisms; o...,supported,Degeneracy wording remains disabled unless Ste...,False
1,R2,"experimental variability, noise, and data cons...",outputs/provenance; outputs/features,supported,Cell/file is the independent unit; no paired a...,False
2,R3,model assumptions and proxy validity,outputs/assumption_sensitivity,partial,Explicit ECS/proxy variants remain required wh...,False
3,R4,Vm-only fits and physiological parameter plaus...,outputs/identifiability; outputs/parameter_pla...,supported,Raw parameters are interpreted only when range...,False
4,R5,"pathways, mechanisms, and phenotypes",outputs/mechanisms; outputs/predictive_validation,supported,Phenotype tags are provisional until predictio...,False
5,R6,held-out prediction and perturbation robustness,outputs/predictive_validation,supported,Prediction-limited or fit-only clusters are no...,False
6,R7,"clarity, organization, units, and figure trace...",outputs/reviewer_synthesis/manuscript_asset_ma...,supported,Figures/tables must cite source step outputs a...,False


## Claim maturity

In [6]:
maturity = result["claim_maturity_table"]
maturity

,claim,maturity,basis,remaining_requirement,mean_biological_description_score
0,accepted cell-specific six-sweep ensembles exist,supported,Step 04 accepted ensemble and held-out-current...,increase accepted/reviewer-facing cell coverag...,0.6167
1,candidate mechanism regimes are biologically i...,restricted_model_phenotype_support,Step 05 mechanisms plus Step 06 phenotype robu...,biological pathway wording still requires assu...,0.6167
2,mechanism or phenotype labels are predictive u...,supported,Step 06 robustness labels and biological_descr...,"broaden support across region, condition, mech...",0.6167
3,model assumptions do not drive the conclusion,quantified_model_dependent_or_unresolved,Step 07 sensitivity plus explicit Step 09 assu...,explicit ECS variant or additional data remain...,0.6167
4,accepted parameters are physiologically interp...,semantic_and_identifiability_blocked,Step 08 range/identifiability plus semantic in...,current ranges are guardrails and fitted coord...,0.6167
5,final biological degeneracy wording is allowed,not_allowed_yet,Integrated Step 03-08 synthesis plus restricte...,"requires mechanism distinction, predictive sup...",0.6167


## Manuscript asset manifest

In [7]:
manifest = result["manuscript_asset_manifest"]
manifest

,step,artifact,purpose,exists,claim_scope
0,Step 00,outputs/provenance/atf_region_condition_invent...,ATF data provenance and region/condition audit,True,reviewer_facing_source_table
1,Step 02,outputs/features/condition_region_sweep_thresh...,Region-aware feature thresholds,True,reviewer_facing_source_table
2,Step 03,outputs/identifiability/effective_parameter_ma...,Effective-parameter and identifiability guardr...,True,reviewer_facing_source_table
3,Step 04,outputs/cell_fits/accepted_cell_ensembles.csv,Accepted six-sweep cell ensembles,True,reviewer_facing_source_table
4,Step 04,outputs/cell_fits/cell_fit_candidates.csv,Full candidate history for audit,True,reviewer_facing_source_table
5,Step 05,outputs/mechanisms/accepted_fit_mechanisms.csv,Hidden-current flux decomposition,True,reviewer_facing_source_table
6,Step 05,outputs/mechanisms/accepted_fit_mechanisms_win...,Windowed local/spatial mechanism characterization,True,reviewer_facing_source_table
7,Step 05,outputs/mechanisms/buffering_phenotype_tags.csv,Provisional phenotype tags,True,reviewer_facing_source_table
8,Step 06,outputs/predictive_validation/robustness_summa...,Prediction and perturbation robustness,True,reviewer_facing_source_table
9,Step 07,outputs/assumption_sensitivity/claim_scope_tab...,Assumption-sensitivity claim scope,True,reviewer_facing_source_table


## Objective synthesis conclusion

The final row below is generated from the actual upstream outputs. If final biological degeneracy wording is not allowed, the manuscript response should keep the claim at mechanism-screen or prediction-limited maturity and state the unresolved layer explicitly.

In [8]:
final_claim = maturity[maturity["claim"].eq("final biological degeneracy wording is allowed")]
display(final_claim)
assert set(traceability["reviewer_id"]) == {"R1", "R2", "R3", "R4", "R5", "R6", "R7"}
assert "final_biological_degeneracy_claim_allowed" in result["analysis_summary"]
print("Step 09 synthesis outputs written under", PROJECT_ROOT / "outputs" / "reviewer_synthesis")

,claim,maturity,basis,remaining_requirement,mean_biological_description_score
5,final biological degeneracy wording is allowed,not_allowed_yet,Integrated Step 03-08 synthesis plus restricte...,"requires mechanism distinction, predictive sup...",0.6167


Step 09 synthesis outputs written under /home/xav/code/astromodel_proving/outputs/reviewer_synthesis


## Restricted claim gates

In [9]:
restricted_join = pd.read_csv(PROJECT_ROOT / "outputs" / "reviewer_synthesis" / "restricted_all_gate_join.csv")
restricted_claims = pd.read_csv(PROJECT_ROOT / "outputs" / "reviewer_synthesis" / "restricted_validation_claims.csv")
assumption_gate = pd.read_csv(PROJECT_ROOT / "outputs" / "reviewer_synthesis" / "assumption_gate_audit.csv")
notebook_screen = pd.read_csv(PROJECT_ROOT / "outputs" / "reviewer_synthesis" / "notebook_update_screen_after_selected_actions.csv")
display(restricted_join.groupby(["validation_label", "restricted_degeneracy_claim_allowed", "blocking_axes"], dropna=False).size().reset_index(name="n_rows"))
display(restricted_claims)
display(assumption_gate)
display(notebook_screen)


,validation_label,restricted_degeneracy_claim_allowed,blocking_axes,n_rows
0,prediction_limited,False,prediction_perturbation;assumptions;parameter_...,7
1,predictive_supported,False,assumptions;parameter_step08;parameter_semantics,2
2,predictive_supported,False,assumptions;parameter_step08;parameter_semanti...,1
3,NaN,False,prediction_perturbation;assumptions;parameter_...,20


,mechanism_cluster,region,condition,validation_label,restricted_all_gate_pass,blocking_axes,allowed_claim,forbidden_claim
0,M1,DH,CONTROL,predictive_supported,False,not_joined,predictive model-derived mechanism scenario,broad biological degeneracy or pathway-level p...
1,M1,DH,MFA,prediction_limited,False,prediction_perturbation;assumptions;parameter_...,prediction-limited mechanism candidate,broad biological degeneracy or pathway-level p...
2,M1,VH,MFA,prediction_limited,False,prediction_perturbation;assumptions;parameter_...,prediction-limited mechanism candidate,broad biological degeneracy or pathway-level p...
3,M2,DH,MFA,prediction_limited,False,prediction_perturbation;assumptions;parameter_...,prediction-limited mechanism candidate,broad biological degeneracy or pathway-level p...
4,M2,VH,CONTROL,prediction_limited,False,not_joined,prediction-limited mechanism candidate,broad biological degeneracy or pathway-level p...
5,M2,VH,MFA,prediction_limited,False,prediction_perturbation;assumptions;parameter_...,prediction-limited mechanism candidate,broad biological degeneracy or pathway-level p...
6,M3,DH,MFA_BA,predictive_supported,False,not_joined,predictive model-derived mechanism scenario,broad biological degeneracy or pathway-level p...
7,M3,VH,MFA_BA,predictive_supported,False,assumptions;parameter_step08;parameter_semanti...,predictive model-derived mechanism scenario,broad biological degeneracy or pathway-level p...


,assumption_axis,n_rows,metric,metric_value,threshold,gate_pass,gate_status,claim_scope
0,gating_form,180,unstable_fraction,0.016667,0.25,True,pass,same-parameter gating-family screen; refit-lev...
1,intracellular_K_as_ECS_proxy,30,proxy_limited_fraction,0.566667,0.25,False,fail,proxy exclusion can restrict claims; explicit ...
2,local_syncytial_compartment_split,30,split_sensitive_fraction,0.000000,0.25,True,pass,lumped split appears stable in current screen ...


,notebook,rerun_required,comment_update_recommended,action_taken,remaining_action,reason
0,outputs/executed_notebooks/00_data_provenance_...,False,False,none_required,none,Selected actions reuse downstream Step 04-08 o...
1,outputs/executed_notebooks/01_feature_extracti...,False,False,none_required,none,Feature thresholds are consumed by the interpo...
2,outputs/executed_notebooks/02_feature_contract...,False,False,none_required,none,Region-aware feature bands are reused unchange...
3,outputs/executed_notebooks/03_combined_identif...,False,False,none_required,none,Step 03 identifiability evidence is reused unc...
4,outputs/executed_notebooks/04_cell_specific_si...,False,False,none_required,none,No Step 04 refit is performed; full accepted e...
5,outputs/executed_notebooks/05_mechanistic_deco...,False,False,interpretation_centralized_in_step09,optional_future_cross_reference_only,New interpolation and phenotype-threshold audi...
6,outputs/executed_notebooks/06_predictive_valid...,False,False,interpretation_centralized_in_step09,optional_future_cross_reference_only,"New phenotype robustness, prediction-limited f..."
7,outputs/executed_notebooks/07_assumption_sensi...,False,False,interpretation_centralized_in_step09,optional_future_cross_reference_only,All-current and proxy-exclusion audits refine ...
8,outputs/executed_notebooks/08_parameter_plausi...,False,False,interpretation_centralized_in_step09,optional_future_cross_reference_only,"Full-ensemble, semantic-class, citation, and c..."
9,outputs/executed_notebooks/09_reviewer_respons...,False,False,updated_and_rerun,none_unless_selected_action_gates_change,Step 09 was updated to incorporate selected-ac...


## Post-execution scientific status

Executed status for reviewer response after selected actions: Step 09 integrates R1-R7 into 7 traceability rows, 6 claim-maturity rows, and a complete expanded manuscript manifest. The selected gate audits produce no-refit/current-output evidence plus modest sensitivity screens. They raise the mechanism-regime claim only to `restricted_model_phenotype_support`, because some model-derived phenotype/stratum gates pass. They do **not** support global biological degeneracy: the restricted all-gate join has no passing row, the intracellular-K proxy assumption gate fails, and parameter semantics/identifiability block direct physiological parameter wording. Final biological degeneracy wording therefore remains `not_allowed_yet`.

Notebook-screen result: core Steps 05-08 do not require computational reruns because their source outputs were reused unchanged, but their interpretation comments can optionally point to the new derivative audits. Step 09 must be rerun whenever these selected-action gates change.